# INFORMACIÓN DEL PROYECTO

**Bootcamp:** Data Science & Inteligencia Artificial

**Tipo de proyecto:** aprendizaje automático aplicado a datos experimentales de baterías.

### TÍTULO

# Predicción de la capacidad y degradación de baterías de ion-litio mediante Machine Learning

### TEMA

El proyecto estudia la evolución de la capacidad de celdas recargables de ion-litio sometidas a distintos perfiles de uso y condiciones experimentales. Se utilizarán mediciones eléctricas y térmicas —tensión, corriente, temperatura y tiempo— para analizar la degradación y evaluar si es posible estimar la capacidad de una batería a partir de su historial de utilización.

El trabajo parte del conjunto **Randomized Battery Usage Data Set (NASA PCoE, dataset n.º 11)**, que contiene siete partes experimentales y 28 baterías identificadas como RW1–RW28.

### OBJETIVOS

**Objetivo general**

Evaluar la viabilidad de desarrollar un modelo de Machine Learning capaz de estimar la capacidad disponible de una celda de ion-litio a partir de las características de su utilización y de las condiciones de operación.

**Objetivos específicos**

1. Inspeccionar y preparar los registros experimentales de las 28 baterías.
2. Identificar las descargas de referencia y calcular su capacidad integrando la corriente respecto al tiempo.
3. Analizar la evolución de la capacidad y su relación con los perfiles de uso y las condiciones experimentales.
4. Construir, si la estructura y cobertura de los datos lo permiten, un conjunto de características de uso previo para estimar la capacidad de la siguiente descarga de referencia.
5. Evaluar modelos de regresión con una validación que mantenga separadas las baterías entre entrenamiento y prueba, evitando fugas de información.

**Variable objetivo inicial:** capacidad estimada de la descarga de referencia, expresada en Ah. El SOH (State of Health) podrá derivarse posteriormente como capacidad actual respecto a una capacidad inicial de referencia, siempre que se valide que ambas mediciones son comparables.

**Alcance:** en esta fase se estudia la viabilidad y se prepara el conjunto de datos. No se afirma todavía que exista un modelo predictivo validado ni una precisión determinada.

## OBTENCIÓN DE LOS DATOS

### DATASETS Y FUENTES

**Dataset principal:** Randomized Battery Usage Data Set (NASA Prognostics Center of Excellence, PCoE), identificado como dataset n.º 11 en el repositorio.

- Fuente institucional: NASA Ames Research Center, Prognostics Center of Excellence.
- Repositorio: https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/
- Descarga del dataset: https://phm-datasets.s3.amazonaws.com/NASA/11.+Randomized+Battery+Usage+Data+Set.zip

**Estructura examinada:** siete partes experimentales, con cuatro baterías por parte (28 en total, RW1–RW28). Los archivos originales están en formato MATLAB `.mat`.

**Variables disponibles en los registros:** `time`, `relativeTime`, `voltage`, `current`, `temperature`, `date`, `comment` y `type`. Los tipos de operación identificados son carga (C), descarga (D) y reposo (R). Los comentarios permiten distinguir distintos perfiles de descarga, incluidas las descargas de referencia.

**Fuente alternativa:** no se ha seleccionado otra fuente por el momento. Se prioriza el uso del dataset experimental de NASA para mantener la trazabilidad de las medidas.

### COMPROBACIÓN PRELIMINAR DE VIABILIDAD

La inspección inicial del conjunto completo ha permitido comprobar:

- 28 archivos de batería identificados y procesados.
- 950 operaciones etiquetadas como `reference discharge`.
- Capacidad calculada para las 950 operaciones mediante integración numérica de la corriente respecto a `relativeTime`.
- Se detectaron tres operaciones con seis mediciones y corriente nula (RW17, RW18 y RW19). Se han marcado como no válidas para el cálculo de capacidad, quedando 947 operaciones tras el filtro provisional.

Estos resultados demuestran que existe una base de datos medible para estudiar la capacidad. No demuestran todavía que todas las operaciones sean comparables ni que se pueda predecir la capacidad con precisión. Queda pendiente validar la cronología, la comparabilidad entre protocolos y la construcción de variables predictoras a partir del uso anterior a cada descarga de referencia.

### VISTA DE LOS DATASETS

Se trabajará principalmente con dos tablas derivadas del dataset original:

**1. `df_baterias` — mediciones consolidadas.** Una fila por medición, identificada por batería, parte experimental, operación y posición de la medición. Incluye las variables eléctricas, térmicas y temporales.

**2. `df_capacidades_validas` — capacidad de referencia.** Una fila por operación de descarga de referencia aceptada por el control de calidad. Incluye batería, parte, número de operación (`step`) y capacidad estimada en Ah.

Las tablas se generan en el notebook `01_inspeccion_datos.ipynb`. Para mostrar sus `head()` en este documento, ejecuta primero el notebook de inspección en el mismo entorno o carga aquí las tablas exportadas.

In [ ]:
from pathlib import Path

salida = Path("../data/processed")
salida.mkdir(parents=True, exist_ok=True)

df_baterias.head(5).to_csv(
    salida / "muestra_baterias.csv", index=False
)

df_capacidades_validas.head(5).to_csv(
    salida / "muestra_capacidades.csv", index=False
)

In [ ]:
df_1 = pd.read_csv("../data/processed/muestra_baterias.csv")
display(df_1.head())

,battery_id,dataset_part,step,measurement,comment,type,date,time,relativeTime,voltage,current,temperature
0,RW10,Battery_Uniform_Distribution_Charge_Discharge_DataSet_2Post,0,0,reference charge,C,06-Jan-2014 13:36:43,6.62,0.01,4.196,-0.000,22.39476
1,RW10,Battery_Uniform_Distribution_Charge_Discharge_DataSet_2Post,0,1,reference charge,C,06-Jan-2014 13:36:43,16.62,10.01,4.200,-0.039,22.41031
2,RW10,Battery_Uniform_Distribution_Charge_Discharge_DataSet_2Post,0,2,reference charge,C,06-Jan-2014 13:36:43,26.62,20.01,4.200,-0.035,22.30144
3,RW10,Battery_Uniform_Distribution_Charge_Discharge_DataSet_2Post,0,3,reference charge,C,06-Jan-2014 13:36:43,36.62,30.01,4.200,-0.033,22.27033
4,RW10,Battery_Uniform_Distribution_Charge_Discharge_DataSet_2Post,0,4,reference charge,C,06-Jan-2014 13:36:43,46.62,40.01,4.200,-0.032,22.30144


In [ ]:
df_2 = pd.read_csv("../data/processed/muestra_capacidades.csv")
display(df_2.head())

,battery_id,dataset_part,step,capacity_ah
0,RW1,Battery_Uniform_Distribution_Variable_Charge_Room_Temp_DataSet_2Post,3,2.000349
1,RW1,Battery_Uniform_Distribution_Variable_Charge_Room_Temp_DataSet_2Post,5,2.000235
2,RW1,Battery_Uniform_Distribution_Variable_Charge_Room_Temp_DataSet_2Post,1009,1.958467
3,RW1,Battery_Uniform_Distribution_Variable_Charge_Room_Temp_DataSet_2Post,1011,1.963174
4,RW1,Battery_Uniform_Distribution_Variable_Charge_Room_Temp_DataSet_2Post,1925,1.911200


### PLAN DE MODELADO PREVISTO

Si la fase de preparación confirma suficientes muestras comparables, se abordará un problema de **regresión**, tomando como objetivo la capacidad de la siguiente descarga de referencia. Las variables predictoras se calcularán exclusivamente con operaciones de uso anteriores a esa descarga, para evitar fuga de información.

Se podrán estudiar modelos de referencia y modelos basados en árboles, por ejemplo Regresión Lineal, Random Forest y Gradient Boosting. La selección final dependerá del tamaño y la calidad del conjunto de entrenamiento.

La evaluación considerará métricas de regresión como MAE, RMSE y R². La separación de entrenamiento y prueba deberá realizarse por batería (por ejemplo, GroupKFold o Leave-One-Group-Out), no mediante una división aleatoria de filas, ya que las mediciones de una misma batería están relacionadas.

Este plan es provisional: no se asignan resultados ni métricas hasta entrenar y evaluar los modelos.